# Using NVIDIA NIM and Reranking to enhance RAG Pipelines

![build.nvidia.com](images/rr_connect_to_rag.png)


In this notebook, we will do a hands on walkthrough of the NVIDIA NIM and NeMo Retriever components to build a simple retrieval augmented generation (RAG) application.  We base this notebook off the [Reranking Technical Blog](https://developer.nvidia.com/blog/enhancing-rag-pipelines-with-re-ranking/) to work through how we can use the following components
* LLama 3.1 8B NIM 
* Llama 3.2 Embedding NIM 
* Llama 3.2 Reranking NIM

We will first build out our application using cloud endpoints via an application that uses the langchain framework.  We will make use of [ChatNVIDIA](https://python.langchain.com/v0.1/docs/integrations/chat/nvidia_ai_endpoints/) that connects to NVIDIA foundation endpoints at build.nvidia.com initially, and then deploy these components locally to show how to run a RAG workflow on local machines.  

This notebook builds upon concepts learned in the Introduction to NIM and LoRA finetuning notebooks.

**Notebook Requirements**:
* Access to a single NVIDIA GPU (A30, A100, H100, L40S, A40)
* Access to build.nvidia.com - After the lab, sign up for access to the NVIDIA NGC and build.nvidia.com to access the models used in this lab.
* Docker
* Langchain
* Notebook is designed to be run from host system, and not within docker container.

## Why use Reranking in RAG Pipelines?

Reranking is a technique used to improve results in RAG based workflows.   The general principle is to use a first pass retrieval model to get a set of candidate documents, and then use a reranking model to select a subset of the best candidates documents to get a better result.  Reranking can significantly boost recall of the RAG by refining the selection of documents and making sure we are selecting the best candidates for the generation task.  Later in the notebook, we will discuss specifically how the reranker is implemented in the NVIDIA NIM framework.

# Use Case
We  will demonstrate RAG pipeline using some information around NVIDIAs Holoscan GPU Streaming SDK.  No prior knowledge of Holoscan is required, here we use some downloaded manuals/repos to show how RAG can provide much better results than just using based LLM generative models.

Data Assets:

*  [Holoscan user guide PDF](https://developer.download.nvidia.com/assets/Clara/Holoscan_SDK_User_Guide_2.2.0.pdf)
*  [Holohub github repo](https://github.com/nvidia-holoscan/holohub.git)

# Environment Setup

**Lab architecture commentary** 

Lets revisit the following architecture diagram to understand how we will be using NIMs for this launchpad workshop.

![build.nvidia.com](images/rr_connect_to_rag.png)

Since our lab supports both A30 and A40, we need to size our lab to A30 which has 24GB of memory vs the 48GB for the A40.  What will will do here is demonstrate the following

* Embedding NIM - running locally on A30/A40
* Reranking NIM - build.nvidia.com hosted
* LLM NIM - build.nvidia.com hosted

In practice, you would run all your NIMs on local hardware but for this lab we will use build.nvidia.com for the LLM NIM.



## Deploy the NIMs locally 
Here will deploy the embedding and reranking NIMs locally run a basic RAG pipeline inference as shown above.  For convenience we have encoded this setup in docker compose yaml files.  This will allow us to run the NIMs locally on our A30/A40 GPUs.  

If you want to see the configuration, see the [docker-compose.yml](docker/docker-compose.yaml) file. 

In [ ]:
# Setup Environment
from dotenv import load_dotenv
import os

# Load the .env file
load_dotenv('./docker/.env', override=True)

print(f"NGC_API_KEY for lab use only     : {os.getenv('NGC_API_KEY')}")
print(f"NVIDIA_API_KEY for lab use only  : {os.getenv('NVIDIA_API_KEY')}")
print(f"Current working dir              : { os.getcwd() }")
print(f"PROJECT_DIR for lab              : {os.getenv('PROJECT_DIR')}")
print(f"LOCAL_NIM_CACHE for lab          : {os.getenv('LOCAL_NIM_CACHE')}")
print(f"locally hosted model for lab     : {os.getenv('NIM_SERVED_MODEL_NAME')}")



In [ ]:
# Stop all prior running containers 
!scripts/stop_containers.sh

In [ ]:
# Start only embed and reranker services

os.chdir('./docker')
!docker compose up -d nim_embed nim_reranker nim_llama_8b
print("waiting 30 seconds to start")
!sleep 30 # wait for services to start
!docker compose ps
os.chdir('..')
# consumes ~16GB GPU RAM

Lets inspect the models we just loaded.  The embed, reranker, and llm models have been configured to use ports 9990, 9991, and 9992 in our docker_compose.yaml file.  For full specification of the API see https://docs.nvidia.com/nim/large-language-models/latest/api-reference.html

In [ ]:
!docker compose -f docker/docker-compose.yaml ps 
# Setup Base URL variables for use with the rest of the notebook
LLM_URL    = "http://localhost:9990/v1"
RERANK_URL = "http://localhost:9991/v1"
EMBED_URL  = "http://localhost:9992/v1"

#LLM
!curl -X 'GET' {LLM_URL}'/models' |jq
#reranker
!curl -X 'GET' {RERANK_URL}'/models' |jq
#embed
!curl -X 'GET' {EMBED_URL}'/models' |jq



In [ ]:
# Setup MODEL Name variables that we will use for the rest of the lab
LLM_MODEL    = os.getenv('NIM_SERVED_MODEL_NAME')
EMBED_MODEL  = "nvidia/llama-3.2-nv-embedqa-1b-v2"
RERANK_MODEL = "nvidia/llama-3.2-nv-rerankqa-1b-v2"
# NVDEV_LLM_MODEL="nvdev/meta/llama-3.1-8b-instruct"

# Data Preparation 

For this project, we will use 3 data sources
* Holoscan User Guide PDF
* Holohub markdown files
* Holohub python files


The Holoscan USER guide PDF has already been downloaded into the data directory.  We will use langchains document_loaders library to extract the text from the PDF.

The holohub github repo has been downloaded locally.  

This example is interesting because it shows how we can incorporate text from pdf, markdown and python files into a single RAG pipeline.  We will use the Holoscan user guide as the base document, and then use the markdown and python files to provide additional context to the RAG pipeline.

In [ ]:
# Load the PDF file into a document which is a list of documents
from langchain_community.document_loaders import PyPDFLoader
pdf_path = 'data/Holoscan_SDK_User_Guide_2.2.0.pdf'
pdf_documents = PyPDFLoader(pdf_path).load()


In [ ]:
# Print a sample of the processed document
import json
print(json.dumps(pdf_documents[101].to_json(), indent=4)) #page_content

# Check the number of pages in the document
print(f"Length of the document : {len(pdf_documents)}")

## Add markdown files from holohub to documents list

In [ ]:
# Use langchain FileSytemBlobLoader to load the markdown files
from langchain_community.document_loaders.blob_loaders import FileSystemBlobLoader
from langchain_core.documents import Document

loader = FileSystemBlobLoader("./utils/holohub", glob ="**/*.md",)
hh_documents = []
for blob in loader.yield_blobs():
    # Extract text content from the blob
    content = blob.as_string()
    # Create a document with the content and metadata
    doc = Document(page_content=content, metadata={"source": blob.source})
    hh_documents.append(doc)

loader = FileSystemBlobLoader("./utils/holohub", glob ="**/*.py",)
for blob in loader.yield_blobs():
    # Extract text content from the blob
    content = blob.as_string()
    # Create a document with the content and metadata
    doc = Document(page_content=content, metadata={"source": blob.source})
    hh_documents.append(doc)



In [ ]:
print(json.dumps(hh_documents[0].to_json(), indent=4)) #page_content

# Check the number of pages in the document
print(f"Number of markdown documents : {len(hh_documents)}")

In [ ]:
# Concatenate the two document lists
documents = pdf_documents + hh_documents

print(f"Total number of documents : {len(documents)}")
# Average document length
avg_doc_length = sum([len(doc.page_content) for doc in documents])/len(documents)
print(f"Average document length : {avg_doc_length} (characters)")

# Split Into chunks

Make sure to pay attention to the chunk_size parameter in TextSplitter. Setting the right chunk size is critical for RAG performance, as much of a RAG pipeline’s success is based on the retrieval step finding the right context for generation. The retrieval step typically examines smaller chunks of the original text rather than all documents.

The entire prompt (retrieved chunks plus the user query) must fit within the LLM’s context window. Don’t specify chunk sizes too big and balance them out with the estimated query size. Experiment with different chunk sizes, but typical values should be 100-600 tokens, depending on the LLM.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=150)
texts = text_splitter.split_documents(documents)

In [ ]:
print(len(texts))
# Print a sample of the processed document
texts[3201].page_content

In [ ]:
# Lets look at a histrogram of the length of the text chunks created by the text splitter
import matplotlib.pyplot as plt
plt.hist([len(text.page_content) for text in texts], bins=20)
plt.xlabel('Length of text chunk')
plt.ylabel('Number of text chunks')
plt.title('Histogram of the length of the text chunks page content')

# Generate Embeddings

Next, generate embeddings using NVIDIA AI Foundation endpoints and save the embeddings to an offline vector store in the /embed directory for future re-use.

For this task, use FAISS, a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM.

For more detail on NVIDIA Embeddings via langchain, see the [NVIDIA Embeddings Documentation](https://python.langchain.com/v0.2/docs/integrations/text_embedding/nvidia_ai_endpoints/)

In [ ]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_community.vectorstores import FAISS
# Pretty print JSON
[print(emb) for emb in NVIDIAEmbeddings().get_available_models()]


Here you can see the available models that we can use.  For this lab we will make use of the Llama 3.2 embedding model.  This model is also accessible via build.nvidia.com and can be locally download (we show this later).

## Setup Langchain to use local NIMs

The key to using local NIM is really simple.  We modify the calls of `NVIDIAEmbeddings()`, `NVIDIARerank()`, and `ChatNVIDIA()` by including a base_url parameter to point to the endpoint of our local NIMs.  We needed to map the ports properly in our docker compose file

For example 
```
NVIDIAEmbeddings(model='nvidia/llama-3.2-nv-embedqa-1b-v2')
```
becomes
```
NVIDIAEmbeddings(base_url="http://localhost:9992/v1", model='nvidia/llama-3.2-nv-embedqa-1b-v2')
```

In [ ]:
# Local NIM version ->
print(f"For embedding using {EMBED_MODEL}")
embeddings = NVIDIAEmbeddings(base_url=EMBED_URL, model=EMBED_MODEL)
db = FAISS.from_documents(texts, embeddings)
#~2-3 min runtime

After the completion of this step, we have our embeddings stored in GPU memory and ready for use in our RAG pipeline.

# Create Base Retriever
In this lab we will create 2 retrievers 
* A simple retriever that fetches 5 similar documents for question and answer generation.
* A reranking retriever that fetches 45 similar documents and pairs down the result to the 5 best most relavant chunks.
  

In [ ]:
# Create a retriever and fetch the top 10 similar documents for a query
base_retriever = db.as_retriever(search_kwargs={"k": 5},search_type="mmr")


In [ ]:
# Basic test of the retreiver
query = "What are the elements of a holoscan pipeline"
docs = base_retriever.invoke(query)
print(f"Number of documents returned : {len(docs)}")
# Print the first document
print(json.dumps(docs[0].to_json(), indent=4))


# Connect to basic RAG Pipeline

Here we will do setup a query about Holoscan and get a basic response from the RAG pipeline.  

In [ ]:
# from langchain import hub
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# See full prompt at https://smith.langchain.com/hub/rlm/rag-prompt
# prompt = hub.pull("rlm/rag-prompt")
# Setup Messages List that will be passed to LangChain LLMs - feel free to modify and experiment!
from langchain.schema import HumanMessage, SystemMessage

prompt = ChatPromptTemplate([
    ("human",   "You are a useful AI chatbot.  Please provide a succinct answer to the question given the following context: {context} | Question : {question}")
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

llm=ChatNVIDIA(model=LLM_MODEL,
                   base_url=LLM_URL, 
                   temperature=0)

qa_chain = (
    {
        "context": base_retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

query = "Show me how to make a holoscan operator in python, that performs a running sum over a cupy float 32 input vector and outputs a float"  # ans page 36 of pdf
response = qa_chain.invoke(query)

# Render output as Markdown
from IPython.display import Markdown, display
display(Markdown(response))

# Add a re-ranking step

![build.nvidia.com](images/rag_workflow.png)

Here we will look at the exact same query, but now add a reranking step as part of our workflow.  This is a GPU-accelerated model optimized for providing a probability score that a given passage contains the information to answer a question. This re-ranks the previously fetched chunks according to which is most relevant using the same query.

You use the NIM as input to the LangChain contextual compression retriever, which improves retrieval by compressing and filtering documents based on the query context before returning them.


In [ ]:
from langchain_nvidia_ai_endpoints import NVIDIARerank
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
 
# @ https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.contextual_compression.ContextualCompressionRetriever.html

# Create a new retriever that fetches 45 documents
rr_vectordb_retriever = db.as_retriever(search_kwargs={"k": 45})
# ?NVIDIARerank
# reranker = NVIDIARerank(top_n=5)

# Use local reranking NIM
reranker = NVIDIARerank(top_n=5, 
                        model=RERANK_MODEL ,
                        base_url=RERANK_URL)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker, base_retriever=rr_vectordb_retriever,search_type="mmr"
)

reranked_chunks = compression_retriever.invoke(query)

In [ ]:
print(len(reranked_chunks))
# Print the first document
print(json.dumps(reranked_chunks[0].to_json(),indent=4))

# Connect Reranker to a RAG Pipeline
![build.nvidia.com](images/rr_connect_to_rag.png)

In addition to using re-ranking independently, you can add it to a RAG pipeline to further enhance responses by ensuring that they use the most relevant chunks for augmenting the original query.

In the cell below, we add our reranker retriever to our LLM chain to perform Q/A over a pdf.

In [ ]:
# from langchain import hub
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate

# See full prompt at https://smith.langchain.com/hub/rlm/rag-prompt
# prompt = hub.pull("rlm/rag-prompt")
# Setup Messages List that will be passed to LangChain LLMs - feel free to modify and experiment!
from langchain.schema import HumanMessage, SystemMessage

prompt = ChatPromptTemplate([
    ("human",   "You are a useful AI chatbot.  Please provide a succinct answer to the question given the following context: {context} | Question : {question}")
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

llm=ChatNVIDIA(model=LLM_MODEL,
                   base_url=LLM_URL, 
                   temperature=0)

# Note here we use the compression_retriever !
qa_chain = (
    {
        "context": compression_retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

query = "Show me how to make a holoscan operator in python, that performs a running sum over a cupy float 32 input vector and outputs a float"  # ans page 36 of pdf
response = qa_chain.invoke(query)

# Render output as Markdown
from IPython.display import Markdown, display
display(Markdown(response))

## Return data sources with answer
In the following example, we use a slighly modified method to build out langchain retriever.  Here we use some builit-in langchain convenience methods to build out our chain using the compression_retriever from the step before.  The response will contain an answer key that we can print out for more information.  For code syntax details see the [Langchain RAG Tutorials Documentation](https://python.langchain.com/v0.2/docs/tutorials/rag/)


In [ ]:
# Slightly Modified example to show how to customize the prompt and display the supporting documents.
# Source example : https://python.langchain.com/v0.2/docs/tutorials/rag/

from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate

# Prompts Matter!  Try this cell with both rag_prompts... which is better ?

rag_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Define concepts in your answers and explain it as "
    "if you were talking to a non expert."
    "\n\n"
    "{context}"
    "\n\n"
    "Question: {input}"
)

# rag_prompt = ( "You are a useful AI chatbot.  Please provide a succinct answer to the question given the following context: {context} | Question : {input}")



prompt = ChatPromptTemplate.from_messages(
    [
        ("human", rag_prompt),
    ]
)

llm=ChatNVIDIA(model=LLM_MODEL,
               base_url=LLM_URL,  
               temperature=0)

question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(compression_retriever, question_answer_chain)
result = rag_chain.invoke({"input": query})

# Display results in Markdown
display(Markdown(result.get("answer")))

print("Supporting Documents")
for d in result.get("context") :
    print(d.metadata)

## Evaluation Commentary
As you can see, the reranker can access more documents and sort them in a way that provides the most relevant documents to your query.  In general, at this point we would want to introduce an evaluation criteria for our RAG system before performing too many more experiments.  This can be done by curating a test Q/A dataset for your particular application, and using an LLM as a judge technique to score which RAG pipeline is working the best.

For an example of how to build an evaluation pipeline see this [instruction guide](https://nvidia.github.io/GenerativeAIExamples/latest/tools/evaluation/index.html)


### Try you own queries, some provided below....

In [ ]:
query="How does scheduling of operators work in holoscan?"

result = rag_chain.invoke({"input": query})
# Display results in Markdown
display(Markdown(result.get("answer")))
print("Supporting Documents")
for d in result.get("context") :
    print(d.metadata)



In [ ]:
query="show me a python hello world program for holoscan"
result = rag_chain.invoke({"input": query})
# Display results in Markdown
display(Markdown(result.get("answer")))
print("Supporting Documents")
for d in result.get("context") :
    print(d.metadata)

In [ ]:
query="show me a holohub example"
result = rag_chain.invoke({"input": query})
# Display results in Markdown
display(Markdown(result.get("answer")))
print("Supporting Documents")
for d in result.get("context") :
    print(d.metadata)

# Notebook Summary
RAG has emerged as a powerful approach, combining the strengths of LLMs and dense vector representations. By using dense vector representations, RAG models can scale efficiently, making them well-suited for large-scale enterprise applications, such as multilingual customer service chatbots and code generation agents.

As LLMs continue to evolve, it is clear that RAG will play an increasingly important role in driving innovation and delivering high-quality, intelligent systems that can understand and generate human-like language.

When building your own RAG pipeline, it’s important to correctly split the vector store documents into chunks by optimizing the chunk size for your specific content and selecting an LLM with a suitable context length. In some cases, complex chains of multiple LLMs may be required. To optimize RAG performance and measure success, use a collection of robust evaluators and metrics.

Next we will discuss how to deploy the NIM and Reranking models locally to run the RAG pipeline on your own machine.

## Supplementary : Configuration of NIMs using Docker Compose
Here we show how to configure the Embedding NIM in docker compose for ease of use.  

These NIMs are accessible via build.nvidia.com and can be downloaded and run locally.  We show how to configure the docker compose file to run the NIMs locally.

![build.png](images/build_rr_emb.png)

We get the pull tags from build.nvidia.com under the retrieval section and add that to our docker compose `image` key in our [Docker compose sample file](docker/docker-compose.yaml)

In [ ]:
# Print out configuration used for nim_embed
!docker compose config nim_embed


Since all the NIMs by default occupy port 8000 we take some care to do port mappings to available ports on the host.  We also map the NIMs to our available GPUs in our setup.